# KOSPI200 압축 랭크앙상블 전략 — 처음부터 끝까지

> **한 줄 요약**: 머신러닝으로 "오늘 KOSPI200 안에서 상대적으로 유망한 8종목"을 골라
> 동일가중 보유하고 10거래일마다 교체하는 전략.
> 2013~2025 검증 성과 **연복리(CAGR) 16.8%** — 5년마다 원금 약 2배 페이스 (KOSPI 연 5.8%의 약 3배).
> (v18 최종: 외국인·기관·**개인** 수급 완전판 반영 — 롤링 5년 중앙값 +81.7%, 최악 5년 구간도 +0.9%)

| 순서 | 내용 |
|---|---|
| 1 | 사용 데이터 — 무엇을 재료로 쓰나 |
| 2 | 문제 정의 — 무엇을 학습하나 (프로젝트의 핵심 발견) |
| 3 | 피처 — 데이터의 어떤 패턴을 배우나 |
| 4 | 학습 방법 — 어떻게 배우고, 어떻게 미래정보를 차단하나 |
| 5 | 투자 전략 — 예측을 어떻게 돈으로 바꾸나 |
| 6 | 백테스트 — 성과 확인 |
| 7 | 검증과 한계 — 이 숫자를 어디까지 믿을 것인가 |

*이 노트북의 모든 코드는 실제로 실행 가능하며, 본 저장소의 검증 스크립트
(`scripts/run_rank_ensemble_v14_9.py`)와 동일한 로직이다.*

## 0. 준비 — Colab에서 실행하는 법

1. Google Drive의 **`MyDrive/kospi200-quant-lab/data/processed/`** 에 아래 두 가지를 업로드
   (저장소의 `data/processed/` 폴더 그대로, 총 약 460MB):
   - `master_panel.parquet`
   - `model_scores/` 폴더 (parquet 18개)
2. 런타임은 기본 CPU면 충분 (전체 실행 약 2~3분).
3. 아래 셀을 실행하면 Drive 접근 권한을 요청한다 — 승인하면 끝.

In [ ]:
# ── 환경 설정 (Colab / 로컬 자동 감지) ──
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import subprocess
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
    import matplotlib.font_manager as fm
    for _f in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
        fm.fontManager.addfont(_f)
    DATA_DIR = Path("/content/drive/MyDrive/kospi200-quant-lab/data/processed")
else:
    _cands = [Path("../../../data/processed"), Path("../data/processed"),
              Path("/Users/sunshine/DEVELOP/repository/kospi200-quant-lab-2010-2025/data/processed")]
    DATA_DIR = next((p for p in _cands if (p / "master_panel.parquet").exists()), _cands[-1])

assert (DATA_DIR / "master_panel.parquet").exists(), f"데이터를 찾을 수 없음: {DATA_DIR}"
print(f"환경: {'Colab' if IN_COLAB else '로컬'} | DATA_DIR = {DATA_DIR}")

## 1. 사용 데이터

세 곳의 공식 출처를 결합한 **마스터 패널** 하나를 사용한다
(`data/processed/v9/master_panel_v9.parquet`, 114컬럼 × 약 124만 행, 2010~2025).

| 출처 | 내용 | 예시 컬럼 |
|---|---|---|
| **KRX** (한국거래소, pykrx) | 일별 시세·거래량·시가총액·**투자자별 순매수**(외국인/기관/개인)·공매도 | `close`, `market_cap`, `foreign_net_pct` |
| **한국은행 ECOS** | 거시지표 — 기준금리, 환율, CPI, M2, 수출 | `macro_usdkrw_chg_20d`, `kospi_volatility_20` |
| **DART** (전자공시) | 분기 재무제표 — 매출, 영업이익, ROE, 부채비율 | `revenue_z`, `debt_ratio_z`, `roe_ttm_z` |

두 가지 원칙이 중요하다:

1. **PIT (Point-In-Time) 유니버스** — "그 날짜에 실제로 KOSPI200에 들어 있던 종목"만 사용한다.
   지금 살아남은 종목만 보면(생존 편향) 성과가 부풀려진다.
2. **공시일 기준 결합** — 재무제표는 분기말이 아니라 **실제 공시된 날짜** 이후에만 사용한다.
   안 그러면 "아직 발표 안 된 실적"을 보고 투자하는 미래정보 누출이 생긴다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PANEL = DATA_DIR / "master_panel.parquet"

panel = pd.read_parquet(PANEL, columns=[
    "date", "ticker", "close", "market_cap", "volume", "log_return",
    "is_pit_universe_kospi200", "is_volume_zero", "kospi_return",
    "foreign_net_pct", "label_fwd_20d",
])
panel["date"] = pd.to_datetime(panel["date"])
print(f"행 수      : {len(panel):,}")
print(f"기간       : {panel['date'].min().date()} ~ {panel['date'].max().date()}")
print(f"종목 수    : {panel['ticker'].nunique()}")
print(f"PIT 편입 행: {(panel['is_pit_universe_kospi200'] == 1).sum():,}")
panel.head(3)

## 2. 문제 정의 — 무엇을 학습하나 (프로젝트의 핵심 발견)

### 실패한 질문: "이 종목이 오를까?"

이 프로젝트는 원래 강화학습(RL) 39개 실험, 방향 분류 모델 등으로
**"특정 종목이 오를지 내릴지"** 를 예측하려 했다. 결과는 전부 실패였다.
단기 방향 예측 정확도(AUC)가 **0.53** — 동전 던지기(0.5)와 사실상 같았다.
주식의 하루하루 등락은 거의 전부 노이즈이기 때문이다.

### 작동한 질문: "오늘 200종목 중 누가 상대적으로 나은가?"

방향은 못 맞혀도 **순서는 어렴풋이 맞힐 수 있다.** 그리고 순서만 맞혀도 돈이 된다:

- 하루에 200종목의 상대 비교 → 하루 200번의 작은 베팅
- 약한 예측력(IC ≈ 0.03~0.05)이라도 **매일 200번씩 수천 일** 쌓이면 통계적으로 유의한 수익
- 이것이 퀀트의 기본 법칙: **성과 ≈ 예측력 × √(베팅 횟수)**

### 그래서 학습 타깃은 "미래 수익률의 순위"

각 날짜마다, 유니버스 안 모든 종목의 **미래 20일(또는 60일) 수익률 순위**를 0~1로 매긴다.
모델은 "수익률이 몇 %일까"가 아니라 **"이 종목이 상위 몇 %일까"** 를 배운다.

In [ ]:
# 타깃 만들기 시연: 어느 하루의 '미래 20일 수익률 순위'
day = panel.loc[(panel["date"] == "2020-06-01")
                & (panel["is_pit_universe_kospi200"] == 1)
                & panel["label_fwd_20d"].notna(),
                ["ticker", "label_fwd_20d"]].copy()
day["target(순위 0~1)"] = day["label_fwd_20d"].rank(pct=True)
day = day.sort_values("target(순위 0~1)", ascending=False)
print(f"2020-06-01 유니버스 {len(day)}종목 — 미래 20일 수익률 상위/하위 3종목:")
print(day.head(3).to_string(index=False))
print("...")
print(day.tail(3).to_string(index=False))
# 모델의 목표: 이 '순위'를 미래를 보지 않고 그날의 정보만으로 맞히기

## 3. 피처 — 데이터의 어떤 패턴을 배우나

서로 성격이 다른 **두 개의 모델**을 만들어 결합한다:

| | 느린 모델 (slow60) | 빠른 모델 (full20) |
|---|---|---|
| 예측 대상 | 미래 **60일** 수익률 순위 | 미래 **20일** 수익률 순위 |
| 피처 수 | 20개 (천천히 변하는 것만) | 29개 (전부) |
| 역할 | 안정적 기조 — 회전율(매매빈도) 낮음 | 빠른 반응 — 최신 정보 반영 |

피처가 표현하는 패턴 (중요도 순):

1. **재무 건전성** — 부채비율, 매출·이익 성장률, ROE의 횡단면 z-score.
   *"재무가 상대적으로 탄탄해지는 기업이 앞선다"* — 실험 전체에서 가장 중요한 그룹.
2. **장기 모멘텀 (12-1)** — 최근 1년 수익률(직전 1개월 제외).
   *"1년간 강했던 종목이 계속 강하다"* — 학계 100년 검증 팩터.
3. **투자자 수급 (외국인·기관·개인, 각 5/20/60일 누적)** — v18에서 완성된 그룹.
   *"외국인·기관이 꾸준히 사는 종목이 앞서고, 개인이 몰리는 종목은 뒤처진다"*
   — 한국 시장 특유의 신호. 특히 개인 수급을 추가하자 수익의 크기는 그대로인데
   **시간적 균일성이 크게 개선**됐다 (롤링 5년 중앙값 +47.7% → +81.7%).
4. **변동성·기술 지표** — RSI, MACD, ATR, 볼린저 위치, 거래량 이상.
5. **시장 국면** — KOSPI 변동성, 환율 변화, 시장 스트레스 플래그.
   *같은 신호도 강세장과 약세장에서 다르게 작동* — 모델이 국면별로 다른 규칙을 배움.

핵심은 이들의 **비선형 결합**이다. 사람이 규칙으로 쓰면
"부채비율 낮고 AND 모멘텀 좋고 AND ..."의 수십 가지 경우의 수를 못 다루지만,
그래디언트 부스팅 트리는 이걸 자동으로 배운다.

In [ ]:
# 피처 생성 (전체 파이프라인과 동일한 로직의 축약판)
def build_features(df):
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
    g = df.groupby("ticker", sort=False)
    close = df["close"].astype("float64")
    df["mom_60"] = close / g["close"].shift(60) - 1.0            # 3개월 모멘텀
    df["mom_12_1"] = g["close"].shift(20) / g["close"].shift(250) - 1.0  # 12-1 모멘텀
    df["foreign_flow_20d"] = (g["foreign_net_pct"]
                              .rolling(20, min_periods=10).sum()
                              .reset_index(level=0, drop=True))   # 외국인 20일 누적
    return df

panel = build_features(panel)
print(panel[["date", "ticker", "mom_60", "mom_12_1", "foreign_flow_20d"]].dropna().head(3))
print()
print("실제 학습에는 위 3개를 포함해 재무 z 12개, 기술지표 9개, 시장 3개 등")
print("느린 모델 20개 / 빠른 모델 29개 피처를 사용한다. (scripts/run_train_rich_v15_9.py 참고)")

## 4. 학습 방법 — 어떻게 배우고, 어떻게 미래정보를 차단하나

### 모델: LightGBM (그래디언트 부스팅 트리)

- 표 형태 데이터에서 신경망보다 강하고, 결측치를 그대로 다루며, 빠르다.
- 설정: 트리 300개, 잎 63개, 학습률 0.05 — 특별한 튜닝 없이 보수적 기본값.

### 시간 검증: Walk-Forward + Purge

백테스트의 최대 함정은 **미래 정보 누출**이다. 두 장치로 차단한다:

```
2013년 예측 ← 2010~2012년 데이터로만 학습
2014년 예측 ← 2010~2013년 데이터로만 학습
   ...                                        ← 매년 확장 (walk-forward)
2025년 예측 ← 2010~2024년 데이터로만 학습

+ Purge: 학습 데이터의 마지막 45~100일을 버림
  (라벨이 '미래 20/60일 수익률'이라 테스트 구간과 겹치는 것을 방지)
```

즉 **모든 성과 수치는 "그 시점에 알 수 없던 정보를 전혀 쓰지 않은" OOS(표본 외) 성과**다.

### 시드 앙상블 + 랭크 평균 (이 프로젝트가 몸으로 배운 교훈)

LightGBM은 랜덤 시드에 따라 결과가 달라진다. 실제로 이 프로젝트에서
**시드 3개짜리 앙상블이 5년 중앙값 +104.9%를 찍었다가, 9개로 늘리자 +45%로 붕괴**한
사건이 있었다 — 우연히 좋은 시드 3개를 골랐던 것("시드 복권").

그래서 최종 방법은:
1. **시드 9개**로 같은 모델을 9번 학습
2. 각 시드의 예측을 **날짜별 순위(rank)로 변환 후 평균** (원값 평균은 분산 큰 시드에 지배됨)
3. 마지막으로 **20일 이동평균**으로 평활화 (신호 안정화 + 매매비용 절감)

In [ ]:
# ── 시연: 2013년 fold 하나를 실제로 학습해 보기 (약 10~30초) ──
import lightgbm as lgb

FEATS = ["mom_60", "mom_12_1", "foreign_flow_20d", "foreign_net_pct", "log_return"]
demo = panel.loc[(panel["is_pit_universe_kospi200"] == 1)
                 & (panel["is_volume_zero"] == 0)
                 & panel["label_fwd_20d"].notna()
                 & panel["mom_12_1"].notna()].copy()
demo["target"] = demo.groupby("date")["label_fwd_20d"].rank(pct=True)

train = demo.loc[demo["date"] < "2012-11-15"]          # purge 45일 반영
test = demo.loc[(demo["date"] >= "2013-01-01") & (demo["date"] <= "2013-12-31")]

model = lgb.LGBMRegressor(objective="regression", num_leaves=63, learning_rate=0.05,
                          n_estimators=300, min_child_samples=500, seed=42, verbosity=-1)
model.fit(train[FEATS], train["target"])
test = test.assign(pred=model.predict(test[FEATS]))

from scipy.stats import spearmanr
ic = test.groupby("date").apply(
    lambda d: spearmanr(d["pred"], d["label_fwd_20d"]).statistic, include_groups=False)
print(f"학습: {len(train):,}행 (2010~2012) → 예측: 2013년 {len(test):,}행")
print(f"2013년 일평균 RankIC: {ic.mean():+.4f}  (0이면 무작위, 0.03 이상이면 운용 가능 수준)")
print("※ 시연용 5개 피처만으로도 양수 — 실제 모델(20~29피처, 9시드)은 IC ~0.05")

In [ ]:
# ── 실제 신호 로드: 저장된 9시드 학습 결과 → 랭크 평균 앙상블 ──
SEEDS = [42, 7, 2024, 1, 2, 3, 4, 5, 6]
SMOOTH = 20   # 20일 평활

lr_mat = panel.pivot_table(index="date", columns="ticker", values="log_return").sort_index()
DATES, COLS = lr_mat.index, lr_mat.columns

def rank_ensemble(tag):
    ranks = []
    for s in SEEDS:
        f = DATA_DIR / f"model_scores/model_scores_{tag}_seed{s}.parquet"
        m = (pd.read_parquet(f)
             .pivot_table(index="date", columns="ticker", values="score")
             .reindex(index=DATES, columns=COLS).ffill(limit=5)
             .rolling(SMOOTH, min_periods=5).mean())
        ranks.append(m.rank(axis=1, pct=True))     # 시드별 순위로 변환
    return sum(ranks) / len(ranks)                  # 순위를 평균

r_slow = rank_ensemble("flow60")   # 느린 모델 (60일 타깃, 수급 완전판 피처)
r_fast = rank_ensemble("flow20")   # 빠른 모델 (20일 타깃, 수급 완전판 피처)
signal = 0.3 * r_slow + 0.7 * r_fast   # ★ 최종 신호 (v18): 느린 30% + 빠른 70%
print(f"신호 행렬: {signal.shape[0]:,}일 × {signal.shape[1]}종목 (9시드 × 2모델 = 18개 학습 결과의 결합)")

## 5. 투자 전략 — 예측을 돈으로 바꾸는 규칙

규칙은 네 줄이면 전부 설명된다:

> 1. 매 시점, 위에서 만든 **신호 점수(느린 모델 30% + 빠른 모델 70%)로 전 종목 순위**를 매긴다.
> 2. **상위 8종목을 12.5%씩 동일가중**으로 보유한다.
> 3. **10거래일(약 2주)마다** 점검하되, 보유 종목이 **32위(=8×4) 밖으로 밀렸을 때만 교체**한다
>    — "버퍼" 규칙. 순위가 7위↔9위를 오가는 종목을 사고팔며 수수료를 낭비하지 않기 위함.
> 4. 매매 비용은 **편도 0.1%(10bp)** 로 차감한다.

왜 이 숫자들인가 — 정직한 답:

- **8종목**: 집중할수록 기대수익↑ 변동성↑. 6종목 이하는 한두 종목 사고에 취약,
  15종목 이상은 수익이 희석됨을 실험으로 확인.
- **버퍼 4배**: 승자를 오래 들고 가게 해서 회전율을 리밸런스당 ~50%로 낮춤.
- 이 값들은 격자 탐색의 결과이므로 **약간의 낙관 편향이 남아 있을 수 있다**
  (7장의 한계 참고). 다만 주변 값(N=8~10, 버퍼 3~4배)들도 모두 CAGR 14% 이상으로,
  특정 값 하나에 매달린 결과는 아니다.

In [ ]:
# ── 백테스트 엔진 (scripts/run_rank_ensemble_v14_9.py 와 동일 로직) ──
EVAL_START = pd.Timestamp("2013-01-01")
N_HOLD, BUF_MULT, H_REBAL, COST = 8, 4, 10, 0.001

inuni = (panel.pivot_table(index="date", columns="ticker", values="is_pit_universe_kospi200")
         .reindex(index=DATES, columns=COLS).fillna(0).astype(bool))
volm = (panel.pivot_table(index="date", columns="ticker", values="volume")
        .reindex(index=DATES, columns=COLS).fillna(0))
mcap = (panel.pivot_table(index="date", columns="ticker", values="market_cap")
        .reindex(index=DATES, columns=COLS).ffill().fillna(0.0))
kospi = panel.groupby("date")["kospi_return"].first().reindex(DATES).fillna(0.0)
lr_f = lr_mat.fillna(0.0)

def backtest(score_mat):
    n = len(DATES)
    nav, prev_w, held = 1.0, {}, []
    daily_nav, daily_dates = [], []
    i = int(np.where(DATES >= EVAL_START)[0][0])
    while i < n - 1:
        elig = inuni.iloc[i].values & (volm.iloc[i].values > 0) & (mcap.iloc[i].values > 0)
        sc = score_mat.iloc[i].values.copy(); sc[~elig] = -np.inf
        valid = [k for k in np.argsort(-sc) if np.isfinite(sc[k])]
        if len(valid) >= N_HOLD:
            rank_of = {k: r for r, k in enumerate(valid)}
            keep = [k for k in held if rank_of.get(k, 9e9) < BUF_MULT * N_HOLD]  # 버퍼
            held = keep + [k for k in valid if k not in keep][:N_HOLD - len(keep)]
            w = {COLS[k]: 1.0 / N_HOLD for k in held}
        else:
            w = prev_w
        turn = sum(abs(w.get(t, 0) - prev_w.get(t, 0)) for t in set(prev_w) | set(w))
        nav *= 1.0 - COST * turn                      # 매매비용 차감
        seg0, end_i = nav, min(i + 1 + H_REBAL, n)
        if end_i > i + 1 and w:
            ci = [COLS.get_loc(c) for c in w]
            wv = np.array(list(w.values()))
            seg = lr_f.iloc[i + 1:end_i].values[:, ci]
            path = (1.0 - wv.sum()) + np.exp(np.cumsum(seg, axis=0)) @ wv   # 정확 복리
            daily_nav.extend((seg0 * path).tolist())
            daily_dates.extend(DATES[i + 1:end_i].tolist())
            nav = seg0 * float(path[-1])
        prev_w = w
        i += H_REBAL
    return pd.Series(daily_nav, index=pd.DatetimeIndex(daily_dates))

nav = backtest(signal)
print(f"백테스트 완료: {nav.index[0].date()} ~ {nav.index[-1].date()}, 최종 NAV {nav.iloc[-1]:.2f}배")

## 6. 백테스트 성과

In [ ]:
# ── 성과 지표 ──
kospi_nav = np.exp(kospi[DATES >= EVAL_START].cumsum())
kospi_nav = pd.Series(kospi_nav.values, index=DATES[DATES >= EVAL_START])

def perf(nav_s, name):
    yrs = (nav_s.index[-1] - nav_s.index[0]).days / 365.25
    cagr = (nav_s.iloc[-1] / nav_s.iloc[0]) ** (1 / yrs) - 1
    mdd = (nav_s / nav_s.cummax() - 1).min()
    r5 = (nav_s / nav_s.shift(1260) - 1).dropna()   # 롤링 5년(1260거래일)
    return {"": name, "누적": f"{(nav_s.iloc[-1] - 1) * 100:+,.0f}%",
            "CAGR": f"{cagr * 100:.2f}%", "5년 기대(CAGR 환산)": f"{((1 + cagr) ** 5 - 1) * 100:+.0f}%",
            "롤링5년 중앙값": f"{r5.median() * 100:+.1f}%", "롤링5년 최악": f"{r5.min() * 100:+.1f}%",
            "MDD": f"{mdd * 100:.1f}%"}

table = pd.DataFrame([perf(nav, "전략 (압축 랭크앙상블)"), perf(kospi_nav, "KOSPI200")])
print(table.to_string(index=False))

In [ ]:
# ── 그림: 누적 성과 & 롤링 5년 수익률 ──
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["axes.unicode_minus"] = False
for f in ["NanumGothic", "AppleGothic", "Malgun Gothic"]:
    if any(f.lower() in x.name.lower() for x in matplotlib.font_manager.fontManager.ttflist):
        matplotlib.rcParams["font.family"] = f
        break

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=False,
                               gridspec_kw={"height_ratios": [2, 1]})
ax1.plot(nav.index, nav.values, label="전략 (top8 랭크앙상블)", lw=1.6)
ax1.plot(kospi_nav.index, kospi_nav.values, label="KOSPI200", lw=1.4, alpha=0.8)
ax1.set_yscale("log"); ax1.set_title("누적 성과 (2013~2025, 로그축, 비용 10bp 차감)")
ax1.legend(); ax1.grid(alpha=0.3)

r5 = (nav / nav.shift(1260) - 1).dropna() * 100
k5 = (kospi_nav / kospi_nav.shift(1260) - 1).dropna() * 100
ax2.plot(r5.index, r5.values, label="전략 롤링 5년 수익률", lw=1.4)
ax2.plot(k5.index, k5.values, label="KOSPI200", lw=1.2, alpha=0.8)
ax2.axhline(100, color="red", ls="--", lw=1, label="+100% (5년 2배)")
ax2.axhline(0, color="gray", lw=0.8)
ax2.set_title("롤링 5년 수익률 (%)"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. 검증과 한계 — 이 숫자를 어디까지 믿을 것인가

### 이 결과가 통과한 검증

| 검증 | 내용 |
|---|---|
| Walk-forward OOS | 모든 예측이 "그 시점에 없던 정보 0%"로 생성됨 |
| Purge | 라벨-테스트 구간 겹침 차단 (45~100일) |
| PIT 유니버스 | 생존 편향 제거 (그 시점 실제 KOSPI200 구성) |
| 9시드 앙상블 | 시드 복권 차단 — 3시드 +104.9%가 9시드에서 +45%로 붕괴하는 것을 잡아냄 |
| 비용 반영 | 편도 10bp, 20bp에서도 CAGR 15%대 유지 |
| 정확 복리 | 과거 발견된 복리 계산 버그(수익 50%pt 왜곡) 수정 로직 사용 |

### 이 과정에서 잡아낸 함정 2개 (교훈)

1. **집중 폭주 착시**: 비중 상한 없는 실험에서 +283%pt 초과수익이 나왔으나,
   확인해 보니 한 종목에 최대 95%가 실린 몰빵의 행운이었다. → 상한/동일가중 필수.
2. **시드 복권**: "5년 +100% 달성" 구성이 나왔으나 시드를 9개로 늘리자 붕괴.
   → 단일(소수) 시드 결과는 절대 보고하지 않는다.

### 남아 있는 한계 — 반드시 알아야 할 것

- **백테스트 ≠ 실거래**: 종가 체결 가정. 실제로는 슬리피지·호가·부분체결로 성과가 깎인다.
- **MDD -55%**: 지수보다 깊은 낙폭. 2020년 코로나급 폭락에서 절반 이상 빠질 수 있는 전략이다.
- **약한 5년 구간 존재**: 롤링 5년 최악 구간은 약 -2%. "언제 시작해도 5년 뒤 2배"가 아니라
  **"장기 평균이 5년당 2배 페이스"** 다. 이 차이를 이해하는 것이 중요하다.
- **잔여 낙관 편향**: 종목 수·버퍼 등 구성 값이 과거 데이터 탐색으로 정해졌으므로
  미래 성과는 표기보다 낮을 것으로 보는 게 안전하다.
- **모멘텀·수급은 공개된 팩터**: 시간이 갈수록 효과가 약화될 수 있다. 연 1회 재검증 권장.

### 재현 스크립트 (저장소)

| 파일 | 내용 |
|---|---|
| `scripts/run_cs_rank_lgbm_v9.py` | 횡단면 랭킹의 예측력 검증 (IC) |
| `scripts/run_train_more_seeds_v13_9.py` | 9시드 모델 학습 |
| `scripts/run_rank_ensemble_v14_9.py` | 최종 전략 검증 (이 노트북과 동일 로직) |
| `report/learning_method_final_v9.md` | 방법론 전체 문서 + 실패 실험 기록 |